# GUI-Model: Mobile UI Transition & Action Predictor

Qwen3-VL-8B-Instruct 기반 모바일 UI 예측 모델 Fine-tuning 파이프라인.

**전체 파이프라인:**
1. **Stage 1: World Modeling** - UI State (XML) + Action → Next UI State (XML)
2. **Stage 2: Action Prediction** - Screenshot + UI State + Task → Action (JSON)
3. **Evaluation** - Test set으로 4-Way 모델 비교 평가
4. **Merge & Upload** - 학습된 모든 모델 Merge 후 HuggingFace Hub 업로드

**모델 정보:**

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Template | qwen3_vl_nothink |
| Vision Tower | Frozen |

**데이터셋 정보:**

| Dataset | Stage | Entries | Description |
|---------|-------|---------|-------------|
| GUI-Model_stage1_train | 1 | ~2,988 | World Modeling (Train 95%) |
| GUI-Model_stage1_test | 1 | ~157 | World Modeling (Test 5%) |
| GUI-Model_stage2_train | 2 | ~2,990 | Action Prediction (Train 95%) |
| GUI-Model_stage2_test | 2 | ~157 | Action Prediction (Test 5%) |
| Images | - | 3,655 | Mobile UI Screenshots (PNG, 공유) |

**4-Way 비교 실험:**

| Exp | Stage 1 | Stage 2 | Base Model | Config |
|-----|---------|---------|------------|--------|
| [Exp-1] | Full FT | — | `Qwen/Qwen3-VL-8B-Instruct` | stage1_full/qwen3_vl_8b_gui.yaml |
| [Exp-2] | — | LoRA FT | `Qwen/Qwen3-VL-8B-Instruct` | stage2_lora/base.yaml |
| [Exp-3] | Full FT → Merge | LoRA FT | `SaFD-00/qwen3-vl-8b-stage1-world-model` | stage2_lora/world_model.yaml |
| [Exp-4] | — | LoRA FT | `trillionlabs/gWorld-8B` | stage2_lora/gworld.yaml |

**Base 비교:** Stage 1/Stage 2 모두 Qwen3-VL-8B-Instruct (Zero-shot) 대비 성능 측정

**Stage 1 학습 설정:** Full FT, LR=2e-5 (cosine, warmup=0.1), batch=64 (2×8×4GPU), A100×4

**Stage 2 공통 설정:** LoRA r=16, α=32, dropout=0.1, LR=5e-5 (cosine, warmup=0.05), batch=32 (2×8×2GPU), RTX 5090×2

**평가 메트릭:**

| Metric | Stage | Description |
|--------|-------|-------------|
| eval_loss / Perplexity | 1 | Next token prediction loss |
| BLEU / ROUGE-L | 1 | 생성 XML vs GT XML 유사도 |
| Element Accuracy | 1 | XML 요소(태그+속성) 일치율 |
| Type Accuracy | 2 | Action type 일치율 |
| Bounds IoU | 2 | Bounding box 겹침 비율 |
| Overall Score | 2 | Type × (0.5×IoU + 0.5×Params) |

## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
import os
from dotenv import load_dotenv

# ============================================================
# === 여기서 프로젝트 루트 경로를 설정하세요 ===
BASE_DIR = "/path/to/GUI-Model"
# ============================================================

BASE_DIR = os.path.abspath(BASE_DIR)
LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")
DATA_DIR = os.path.join(BASE_DIR, "data")

os.chdir(BASE_DIR)
load_dotenv(os.path.join(BASE_DIR, ".env"))

print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"HF_TOKEN: {'✓ loaded' if os.environ.get('HF_TOKEN') else '✗ not found'}")

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/hiyouga/LlamaFactory.git

In [ ]:
os.chdir(LF_ROOT)
!pip install -e ".[torch,metrics]"
!pip install -r requirements/metrics.txt -r requirements/deepspeed.txt
!pip install vllm
!pip install beautifulsoup4 munkres lxml

## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 Train/Test Split 후 LLaMA-Factory에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: 3,655개 모바일 UI 스크린샷

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage1.jsonl | 3,145 | World Modeling 원본 (Filtered) |
| gui-model_stage1_train.jsonl | ~2,988 | Train Split (95%) |
| gui-model_stage1_test.jsonl | ~157 | Test Split (5%) |

**Split 전략:**
- Random Split (seed=42, 재현 가능)
- 비율: 0.95 : 0.05

In [ ]:
import json
import random
import shutil
from pathlib import Path

# === Paths ===
DATA_PATH = Path(DATA_DIR)
LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. 데이터 디렉토리 생성 ===
LF_GUI_DATA.mkdir(parents=True, exist_ok=True)
(LF_GUI_DATA / "images").mkdir(exist_ok=True)

# === 2. Train/Test Split (0.95:0.05) ===
stage1_path = DATA_PATH / "gui-model_stage1.jsonl"
train_path = DATA_PATH / "gui-model_stage1_train.jsonl"
test_path = DATA_PATH / "gui-model_stage1_test.jsonl"

with open(stage1_path, 'r') as f:
    entries = [json.loads(line) for line in f if line.strip()]

random.seed(42)
random.shuffle(entries)
split_idx = int(len(entries) * 0.95)
train_entries = entries[:split_idx]
test_entries = entries[split_idx:]

for path, data in [(train_path, train_entries), (test_path, test_entries)]:
    with open(path, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"[Split] Total: {len(entries)} → Train: {len(train_entries)}, Test: {len(test_entries)}")
print(f"  Ratio: {len(train_entries)/len(entries):.3f} : {len(test_entries)/len(entries):.3f}")

# === 3. JSONL 파일 복사 ===
for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
    src = DATA_PATH / fname
    dst = LF_GUI_DATA / fname
    if src.exists():
        shutil.copy2(src, dst)
        with open(src, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[Copy] {fname} ({count} entries, {src.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"[WARN] Not found: {src}")

# === 4. 이미지 복사 ===
src_images = DATA_PATH / "images"
dst_images = LF_GUI_DATA / "images"

if src_images.exists():
    img_count = len(list(src_images.glob("*.png")))
    existing = len(list(dst_images.glob("*.png")))
    if existing >= img_count:
        print(f"[Skip] Images already exist ({existing} files)")
    else:
        for img in src_images.glob("*.png"):
            dst_img = dst_images / img.name
            if not dst_img.exists():
                shutil.copy2(img, dst_img)
        print(f"[Copy] {img_count} images")
else:
    print(f"[WARN] Image directory not found: {src_images}")

# === 5. dataset_info.json 등록 ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

DATASETS_TO_REGISTER = {
    "GUI-Model_stage1_train": {
        "file_name": "GUI-Model/gui-model_stage1_train.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    },
    "GUI-Model_stage1_test": {
        "file_name": "GUI-Model/gui-model_stage1_test.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    }
}

for name, config in DATASETS_TO_REGISTER.items():
    dataset_info[name] = config
    print(f"[Registered] {name}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 1 Dataset Statistics ===\n")

for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()
        entry_count = len(lines)

        sample = json.loads(lines[0])
        msg_count = len(sample["messages"])
        img_count = len(sample.get("images", []))

        print(f"[{fname}]")
        print(f"  Entries: {entry_count}")
        print(f"  Sample - Messages: {msg_count}, Images: {img_count}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
        print()

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

## 2. Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 **Train/Test Split** 후 LLaMA-Factory에 등록합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1과 동일 (공유)

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage2.jsonl | 3,147 | Action Prediction (원본) |
| gui-model_stage2_train.jsonl | ~2,990 | Train Split (95%) |
| gui-model_stage2_test.jsonl | ~157 | Test Split (5%) |

**Split 전략:**
- Stratified Split (action type 비율 유지)
- Random seed: 42 (재현 가능)

In [ ]:
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict, Counter

# === Paths ===
DATA_PATH = Path(DATA_DIR)
LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. 데이터 디렉토리 확인 (Stage 1에서 이미 생성됨) ===
LF_GUI_DATA.mkdir(parents=True, exist_ok=True)

# === 2. Stratified Train/Test Split (0.95:0.05, action type별) ===
stage2_path = DATA_PATH / "gui-model_stage2.jsonl"
train_path = DATA_PATH / "gui-model_stage2_train.jsonl"
test_path = DATA_PATH / "gui-model_stage2_test.jsonl"

with open(stage2_path, 'r') as f:
    entries = [json.loads(line) for line in f if line.strip()]

# Action type별 그룹화
type_groups = defaultdict(list)
for entry in entries:
    action = json.loads(entry['messages'][-1]['value'])
    action_type = action.get('type', 'unknown')
    type_groups[action_type].append(entry)

print(f"[Load] Total: {len(entries)}, Action types: {dict(Counter({k: len(v) for k, v in type_groups.items()}))}")

random.seed(42)
train_entries, test_entries = [], []

for action_type, group in type_groups.items():
    random.shuffle(group)
    split_idx = max(1, int(len(group) * 0.95))
    train_entries.extend(group[:split_idx])
    test_entries.extend(group[split_idx:])

random.shuffle(train_entries)
random.shuffle(test_entries)

for path, data in [(train_path, train_entries), (test_path, test_entries)]:
    with open(path, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"[Split] Total: {len(entries)} → Train: {len(train_entries)}, Test: {len(test_entries)}")
print(f"  Ratio: {len(train_entries)/len(entries):.3f} : {len(test_entries)/len(entries):.3f}")

# === 3. JSONL 파일 복사 ===
for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
    src = DATA_PATH / fname
    dst = LF_GUI_DATA / fname
    if src.exists():
        shutil.copy2(src, dst)
        with open(src, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[Copy] {fname} ({count} entries, {src.stat().st_size / 1024 / 1024:.1f} MB)")

# === 4. 이미지 심볼릭 링크 (Stage 1에서 이미 생성됨) ===
src_images = DATA_PATH / "images"
dst_images = LF_GUI_DATA / "images"

if dst_images.exists() or dst_images.is_symlink():
    print(f"[Skip] Image symlink already exists: {dst_images}")
else:
    if src_images.exists():
        dst_images.symlink_to(src_images.resolve())
        imgs = list(src_images.glob("*.png"))
        print(f"[Link] {src_images} → {dst_images} ({len(imgs)} images)")
    else:
        print("[WARN] Image directory not found. Run Stage 1 data preparation first.")

# === 5. dataset_info.json 등록 ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

DATASETS_TO_REGISTER = {
    "GUI-Model_stage2_train": {
        "file_name": "GUI-Model/gui-model_stage2_train.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    },
    "GUI-Model_stage2_test": {
        "file_name": "GUI-Model/gui-model_stage2_test.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    }
}

for name, config in DATASETS_TO_REGISTER.items():
    dataset_info[name] = config
    print(f"[Registered] {name}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path
from collections import Counter

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 2 Dataset Statistics ===\n")

for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()

        print(f"[{fname}]")
        print(f"  Entries: {len(lines)}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

        # Action type 분포
        action_types = Counter()
        for line in lines:
            entry = json.loads(line)
            gpt_val = json.loads(entry['messages'][-1]['value'])
            action_types[gpt_val.get('type', 'unknown')] += 1

        print(f"  Action Type Distribution:")
        for atype, count in action_types.most_common():
            print(f"    {atype}: {count} ({count/len(lines)*100:.1f}%)")
        print()
    else:
        print(f"[WARN] Not found: {fpath}")

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

In [ ]:
import json
from pathlib import Path
from collections import Counter

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 2 Dataset Statistics ===\n")

for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()

        print(f"[{fname}]")
        print(f"  Entries: {len(lines)}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

        # Action type 분포
        action_types = Counter()
        for line in lines:
            entry = json.loads(line)
            gpt_val = json.loads(entry['messages'][-1]['value'])
            action_types[gpt_val.get('type', 'unknown')] += 1

        print(f"  Action Type Distribution:")
        for atype, count in action_types.most_common():
            print(f"    {atype}: {count} ({count/len(lines)*100:.1f}%)")
        print()
    else:
        print(f"[WARN] Not found: {fpath}")

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

## 3. Stage 1 SFT Training

Qwen3-VL-8B-Instruct 베이스 모델을 Stage 1 World Modeling 데이터로 Full Fine-tuning합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**학습 설정:**

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Method | Full Fine-tuning |
| Vision Tower | Frozen |
| Dataset | GUI-Model_stage1_train (~2,988건) |
| Template | qwen3_vl_nothink |
| Cutoff Length | 8,192 |
| Epochs | 3.0 |
| Learning Rate | 2e-5 (cosine, warmup=0.1) |
| Effective Batch | 64 (2 × 8 × 4 GPU) |
| Precision | bf16 |
| DeepSpeed | ZeRO Stage 3 |
| Hardware | A100 80GB × 4 |

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage1_full"
yaml_dir.mkdir(parents=True, exist_ok=True)

STAGE1_CONFIG = """\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_train
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_full/full_world_model
logging_steps: 1
save_strategy: "no"
# save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
learning_rate: 2.0e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
deepspeed: examples/deepspeed/ds_z3_config.json

# ### eval
# val_size: 0.1
# per_device_eval_batch_size: 1
# eval_strategy: steps
# eval_steps: 50
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## [Exp-1] Full Fine-tuning (cosine, lr=2e-5, DeepSpeed Z3) | A100 x 4
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=4 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage1_full/qwen3_vl_8b_gui.yaml

## 4. Stage 1 Merge & Upload to HuggingFace

학습된 Stage 1 LoRA 어댑터를 기본 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**업로드 대상:** `SaFD-00/qwen3-vl-8b-stage1-world-model`

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "gui"
yaml_dir.mkdir(parents=True, exist_ok=True)

MERGE_STAGE1_CONFIG = """\
### model
model_name_or_path: ./outputs/stage1_full/full_world_model
trust_remote_code: true
template: qwen3_vl_nothink

### export
export_dir: ./exports/qwen3-vl-8b-stage1-world-model
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: SaFD-00/qwen3-vl-8b-stage1-world-model
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(MERGE_STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## LoRA Merge & HuggingFace Hub Upload
!llamafactory-cli export examples/merge_custom/GUI-Model/gui/qwen3_vl_8b_gui.yaml

## 5. Stage 1 Evaluation Pipeline

학습된 Stage 1 모델의 World Modeling 성능을 평가합니다.
- **Base (Zero-shot):** `Qwen/Qwen3-VL-8B-Instruct`
- **Exp-1 (Fine-tuned):** `SaFD-00/qwen3-vl-8b-stage1-world-model`

### 5A. Eval-Loss (Intrinsic Evaluation)

Base(Zero-shot) 모델과 Fine-tuned 모델의 다음 토큰 예측 능력을 비교합니다.

| Metric | Description |
|--------|-------------|
| eval_loss | Test set에 대한 Cross-entropy loss |
| Perplexity | exp(eval_loss) - 낮을수록 좋음 |

### 5B. Hungarian Matching (Extrinsic Evaluation)

모델이 생성한 XML과 정답 XML을 비교합니다.

| Metric | Description |
|--------|-------------|
| BLEU-4 | n-gram 기반 텍스트 유사도 |
| ROUGE-L | Longest Common Subsequence F1 |
| Exact Match | GT XML과 완전 일치 비율 |
| Hungarian EA | Element Accuracy (매칭수 / max(pred, gt)) |
| Hungarian F1 | Precision-Recall F1 Score |
| Hungarian Prec | Precision (매칭수 / pred 요소 수) |
| Hungarian Rec | Recall (매칭수 / gt 요소 수) |
| Hungarian Text | 매칭 쌍의 Jaccard 텍스트 유사도 평균 |
| Hungarian Idx | 매칭 쌍의 index 위치 정확도 (|diff| ≤ 2) |

**데이터:** GUI-Model_stage1_test (~157건, 5% split)

In [ ]:
from pathlib import Path

# === Stage 1 Evaluation YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage1_eval"
yaml_dir.mkdir(parents=True, exist_ok=True)

# === 5A. Eval-Loss: Base (Zero-shot) YAML ===
BASE_EVAL_LOSS_CONFIG = """\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_eval: true
finetuning_type: full
freeze_vision_tower: true

### dataset
eval_dataset: GUI-Model_stage1_test
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_eval/eval_loss/base
overwrite_output_dir: true

### eval
per_device_eval_batch_size: 1
"""

fpath = yaml_dir / "base_eval_loss.yaml"
with open(fpath, 'w') as f:
    f.write(BASE_EVAL_LOSS_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

# === 5A. Eval-Loss: Exp-1 (Fine-tuned) YAML ===
EVAL_LOSS_CONFIG = """\
### model
model_name_or_path: SaFD-00/qwen3-vl-8b-stage1-world-model
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_eval: true
finetuning_type: full
freeze_vision_tower: true

### dataset
eval_dataset: GUI-Model_stage1_test
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_eval/eval_loss/exp1
overwrite_output_dir: true

### eval
per_device_eval_batch_size: 1
"""

fpath = yaml_dir / "eval_loss.yaml"
with open(fpath, 'w') as f:
    f.write(EVAL_LOSS_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

# === 5B. Hungarian Matching: Base (Zero-shot) YAML ===
(yaml_dir / "base").mkdir(exist_ok=True)

BASE_PREDICT_CONFIG = """\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_predict: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_test
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_eval/hungarian_matching/base
overwrite_output_dir: true

### predict
per_device_eval_batch_size: 1
predict_with_generate: true
"""

fpath = yaml_dir / "base" / "predict.yaml"
with open(fpath, 'w') as f:
    f.write(BASE_PREDICT_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

# === 5B. Hungarian Matching: Full World Model (Fine-tuned) YAML ===
(yaml_dir / "full_world_model").mkdir(exist_ok=True)

FWM_PREDICT_CONFIG = """\
### model
model_name_or_path: ./outputs/stage1_full/full_world_model
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_predict: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_test
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_eval/hungarian_matching/full_world_model
overwrite_output_dir: true

### predict
per_device_eval_batch_size: 1
predict_with_generate: true
"""

fpath = yaml_dir / "full_world_model" / "predict.yaml"
with open(fpath, 'w') as f:
    f.write(FWM_PREDICT_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## Stage 1 - 5A. Eval-Loss: Base (Zero-shot)
!llamafactory-cli train examples/train_custom/GUI-Model/stage1_eval/base_eval_loss.yaml

## Stage 1 - 5A. Eval-Loss: Exp-1 (Fine-tuned)
!llamafactory-cli train examples/train_custom/GUI-Model/stage1_eval/eval_loss.yaml

In [ ]:
import json, math, os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from datetime import datetime

# === 5A. Eval-Loss 결과 시각화 및 리포트 (Base vs Exp-1 비교) ===
eval_loss_dir = Path(LF_ROOT) / "outputs" / "stage1_eval" / "eval_loss"

# --- 두 모델 결과 로드 ---
base_result_path = eval_loss_dir / "base" / "eval_results.json"
exp1_result_path = eval_loss_dir / "exp1" / "eval_results.json"

with open(base_result_path, 'r') as f:
    base_loss_data = json.load(f)
with open(exp1_result_path, 'r') as f:
    exp1_loss_data = json.load(f)

base_eval_loss = base_loss_data.get('eval_loss', 0)
base_perplexity = math.exp(base_eval_loss)
exp1_eval_loss = exp1_loss_data.get('eval_loss', 0)
exp1_perplexity = math.exp(exp1_eval_loss)

print(f"Base (Zero-shot)   eval_loss: {base_eval_loss:.4f}  perplexity: {base_perplexity:.4f}")
print(f"Exp-1 (Fine-tuned) eval_loss: {exp1_eval_loss:.4f}  perplexity: {exp1_perplexity:.4f}")

# --- 시각화: 2-panel bar chart (Base vs Exp-1) ---
labels = ['Base\n(Zero-shot)', 'Exp-1\n(Fine-tuned)']
colors = ['#9E9E9E', '#FF5722']

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Eval Loss 비교
losses = [base_eval_loss, exp1_eval_loss]
bars0 = axes[0].bar(labels, losses, color=colors, width=0.5)
axes[0].set_title('5A. Eval Loss')
axes[0].set_ylabel('Cross-Entropy Loss')
for bar, val in zip(bars0, losses):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + max(losses)*0.02,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, max(losses) * 1.3)
axes[0].grid(axis='y', alpha=0.3)

# Perplexity 비교
perps = [base_perplexity, exp1_perplexity]
bars1 = axes[1].bar(labels, perps, color=colors, width=0.5)
axes[1].set_title('5A. Perplexity')
axes[1].set_ylabel('Perplexity = exp(loss)')
for bar, val in zip(bars1, perps):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + max(perps)*0.02,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, max(perps) * 1.3)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Stage 1 - 5A. Eval-Loss (Intrinsic Evaluation)', fontsize=14, fontweight='bold')
plt.tight_layout()
img_path = eval_loss_dir / "stage1_eval_loss_evaluation.png"
plt.savefig(img_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {img_path}")

# --- Markdown 리포트 (비교 테이블) ---
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
loss_improve = (base_eval_loss - exp1_eval_loss) / base_eval_loss if base_eval_loss else 0
ppl_improve = (base_perplexity - exp1_perplexity) / base_perplexity if base_perplexity else 0

report = f"""# Stage 1 - 5A. Eval-Loss Report: Base vs Exp-1

> Generated: {now}

## Experiment Setup

| Item | Value |
|------|-------|
| Base Model | Qwen/Qwen3-VL-8B-Instruct (Zero-shot) |
| Exp-1 Model | SaFD-00/qwen3-vl-8b-stage1-world-model (Fine-tuned) |
| Test Dataset | GUI-Model_stage1_test (~157 samples) |
| Task | World Modeling (UI State XML + Action → Next UI State XML) |
| Evaluation | Cross-entropy loss on held-out test set |

## Loss Metrics Comparison

| Metric | Base (Zero-shot) | Exp-1 (Fine-tuned) | Improvement |
|--------|-----------------|---------------------|-------------|
| Eval Loss | {base_eval_loss:.4f} | {exp1_eval_loss:.4f} | {loss_improve:.1%} ↓ |
| Perplexity | {base_perplexity:.4f} | {exp1_perplexity:.4f} | {ppl_improve:.1%} ↓ |

## Metric Definitions

| Metric | Description |
|--------|-------------|
| Eval Loss | 테스트 셋에 대한 평균 Cross-entropy Loss. 모델이 정답 토큰에 부여한 확률의 음의 로그 평균 |
| Perplexity | exp(eval_loss). 모델이 다음 토큰을 예측할 때의 혼란도 (1에 가까울수록 확신이 높음) |
"""

report_path = eval_loss_dir / "evaluation_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"[Saved] {report_path}")

In [ ]:
os.chdir(LF_ROOT)
!mkdir -p ./outputs/stage1_eval/hungarian_matching/base

## Stage 1 - 5B. Hungarian Matching: Baseline (Qwen3-VL-8B Zero-shot)
!python scripts/vllm_infer.py \
    --model_name_or_path Qwen/Qwen3-VL-8B-Instruct \
    --dataset GUI-Model_stage1_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage1_eval/hungarian_matching/base/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage1_eval/hungarian_matching/base/predict_results.json

In [ ]:
os.chdir(LF_ROOT)
!mkdir -p ./outputs/stage1_eval/hungarian_matching/full_world_model

## Stage 1 - 5B. Hungarian Matching: Exp-1 (Fine-tuned)
!python scripts/vllm_infer.py \
    --model_name_or_path SaFD-00/qwen3-vl-8b-stage1-world-model \
    --dataset GUI-Model_stage1_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage1_eval/hungarian_matching/full_world_model/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage1_eval/hungarian_matching/full_world_model/predict_results.json

In [ ]:
import json
import re
import math
from collections import Counter
from bs4 import BeautifulSoup
from munkres import Munkres

# ── Hungarian Metric 상수 ─────────────────────────────────────────────────
INTERACTIVE_TAGS = {"button", "input", "a", "select", "textarea"}
CONTENT_TAGS     = {"p", "img", "span"}
CLICKABLE_ATTRS  = {"clickable", "long-clickable"}

W_TAG   = 3.0   # tag 불일치 패널티 (매칭 불가)
W_TEXT  = 1.5   # text 불일치
W_INDEX = 0.2   # DOM index 거리

MATCH_THRESHOLD = 1.5   # 이 이상이면 매칭 거부
INDEX_TAU       = 2     # index_acc: 차이 ≤ τ 이면 위치 정확


# ── 요소 추출 (BeautifulSoup) ─────────────────────────────────────────────

def _collect_texts(el):
    """요소 자신 + 자식 전체에서 텍스트 토큰 수집 (중복 제거, 알파벳순 join)."""
    tokens = set()
    def add(v):
        if v:
            tokens.add(v.strip())
    add(el.get("description"))
    add(el.get("id"))
    for child in el.find_all(True):
        add(child.get("description"))
        add(child.get("id"))
        t = child.get_text(strip=True)
        if t:
            tokens.add(t)
    t = el.get_text(strip=True)
    if t:
        tokens.add(t)
    return " | ".join(sorted(tokens)) if tokens else ""


def _safe_int(v, default=-1):
    try:
        return int(v)
    except (TypeError, ValueError):
        return default


def extract_elements(xml_str):
    """XML/HTML 문자열에서 interactive 요소를 평탄화하여 추출."""
    try:
        soup = BeautifulSoup(xml_str, "xml")
    except Exception:
        soup = BeautifulSoup(xml_str, "html.parser")
    elements = []
    for el in soup.find_all(True):
        tag  = el.name
        idx  = _safe_int(el.get("index", -1))
        text = _collect_texts(el)
        is_interactive = tag in INTERACTIVE_TAGS
        is_content     = (tag in CONTENT_TAGS) and bool(text)
        is_clickable   = any(el.get(a) for a in CLICKABLE_ATTRS)
        if is_interactive or is_content or is_clickable:
            elements.append({"tag": tag, "text": text, "index": idx})
    return elements


# ── 비용 함수 ─────────────────────────────────────────────────────────────

def _text_sim(a, b):
    """Jaccard 유사도 (단어 집합 기준)."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    sa = set(a.lower().replace("|", "").split())
    sb = set(b.lower().replace("|", "").split())
    return len(sa & sb) / len(sa | sb)


def _match_cost(e1, e2, max_idx):
    """pred 요소 e1과 gt 요소 e2 사이의 매칭 비용."""
    if e1["tag"] != e2["tag"]:
        return W_TAG
    tc = W_TEXT  * (1.0 - _text_sim(e1["text"], e2["text"]))
    ic = W_INDEX * (abs(e1["index"] - e2["index"]) / max(max_idx, 1))
    return round(tc + ic, 5)


# ── 헝가리안 매칭 ─────────────────────────────────────────────────────────

def _hungarian_match(pred, gt):
    """헝가리안 알고리즘으로 pred-gt 간 최적 1:1 매칭."""
    n, m = len(pred), len(gt)
    if n == 0 or m == 0:
        return [], []
    max_idx = max(
        (e["index"] for e in pred + gt if e["index"] >= 0),
        default=1,
    )
    matrix = [
        [_match_cost(p, g, max_idx) for g in gt]
        for p in pred
    ]
    size   = max(n, m)
    padded = [row + [MATCH_THRESHOLD * 2] * (size - len(row)) for row in matrix]
    while len(padded) < size:
        padded.append([MATCH_THRESHOLD * 2] * size)
    indexes = Munkres().compute(padded)
    pairs = []
    for i, j in indexes:
        if i < n and j < m and matrix[i][j] < MATCH_THRESHOLD:
            pairs.append((i, j, matrix[i][j]))
    return pairs, matrix


def compute_hungarian_acc(pred_str, gt_str):
    """모델 생성 XML과 정답 XML을 비교해 hungarian 기반 평가 메트릭을 반환."""
    _zero = {
        "hungarian_ea": 0.0, "hungarian_f1": 0.0,
        "hungarian_prec": 0.0, "hungarian_rec": 0.0,
        "hungarian_text": 0.0, "hungarian_idx": 0.0,
    }
    try:
        pred_els = extract_elements(pred_str)
        gt_els   = extract_elements(gt_str)
    except Exception:
        return _zero
    if not gt_els:
        return _zero

    pairs, _ = _hungarian_match(pred_els, gt_els)
    n_pred, n_gt, n_matched = len(pred_els), len(gt_els), len(pairs)

    ea   = n_matched / max(n_pred, n_gt) if max(n_pred, n_gt) > 0 else 0.0
    prec = n_matched / n_pred             if n_pred  > 0           else 0.0
    rec  = n_matched / n_gt               if n_gt    > 0           else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0    else 0.0

    if pairs:
        text_sims = [_text_sim(pred_els[i]["text"], gt_els[j]["text"]) for i, j, _ in pairs]
        idx_diffs = [abs(pred_els[i]["index"] - gt_els[j]["index"]) for i, j, _ in pairs]
        text_avg  = sum(text_sims) / len(text_sims)
        idx_acc   = sum(1 for d in idx_diffs if d <= INDEX_TAU) / len(idx_diffs)
    else:
        text_avg = 0.0
        idx_acc  = 0.0

    return {
        "hungarian_ea":   round(ea, 4),
        "hungarian_f1":   round(f1, 4),
        "hungarian_prec": round(prec, 4),
        "hungarian_rec":  round(rec, 4),
        "hungarian_text": round(text_avg, 4),
        "hungarian_idx":  round(idx_acc, 4),
    }


# ── 텍스트 생성 품질 메트릭 ───────────────────────────────────────────────

def calc_bleu(reference, hypothesis, max_n=4):
    """BLEU-4 score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not hyp_tokens or not ref_tokens:
        return 0.0

    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1))

        clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())

        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    return bp * math.exp(log_avg)


def calc_rouge_l(reference, hypothesis):
    """ROUGE-L (F1) score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_len = dp[m][n]
    precision = lcs_len / n
    recall = lcs_len / m

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ── Stage 1 전체 평가 함수 ────────────────────────────────────────────────

def evaluate_stage1_predictions(test_path, pred_path):
    """Stage 1 prediction 전체 평가를 수행합니다."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_text = gt_entry['messages'][-1]['value']
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))

        bleu = calc_bleu(gt_text, pred_text)
        rouge_l = calc_rouge_l(gt_text, pred_text)
        exact_match = 1.0 if gt_text.strip() == pred_text.strip() else 0.0
        hungarian = compute_hungarian_acc(pred_text, gt_text)

        results.append({
            'bleu': bleu, 'rouge_l': rouge_l,
            'exact_match': exact_match, 'hungarian': hungarian,
        })

    total = len(results)
    metrics = {
        'total': total,
        'avg_bleu': sum(r['bleu'] for r in results) / total if total else 0,
        'avg_rouge_l': sum(r['rouge_l'] for r in results) / total if total else 0,
        'exact_match_rate': sum(r['exact_match'] for r in results) / total if total else 0,
        'avg_hungarian_ea':   sum(r['hungarian']['hungarian_ea'] for r in results) / total if total else 0,
        'avg_hungarian_f1':   sum(r['hungarian']['hungarian_f1'] for r in results) / total if total else 0,
        'avg_hungarian_prec': sum(r['hungarian']['hungarian_prec'] for r in results) / total if total else 0,
        'avg_hungarian_rec':  sum(r['hungarian']['hungarian_rec'] for r in results) / total if total else 0,
        'avg_hungarian_text': sum(r['hungarian']['hungarian_text'] for r in results) / total if total else 0,
        'avg_hungarian_idx':  sum(r['hungarian']['hungarian_idx'] for r in results) / total if total else 0,
    }
    return metrics, results

print("[Loaded] Stage 1 evaluation functions: calc_bleu, calc_rouge_l, compute_hungarian_acc, evaluate_stage1_predictions")

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path

test_path = Path(LF_ROOT) / "data" / "GUI-Model" / "gui-model_stage1_test.jsonl"

# === Stage 1 평가: Baseline (Zero-shot) vs Exp-1 (Fine-tuned) ===

MODEL_PREDS = {
    "Baseline (Zero-shot)": Path(LF_ROOT) / "outputs" / "stage1_eval" / "hungarian_matching" / "base" / "generated_predictions.jsonl",
    "Exp-1 (Fine-tuned)": Path(LF_ROOT) / "outputs" / "stage1_eval" / "hungarian_matching" / "full_world_model" / "generated_predictions.jsonl",
}

# === 1. Loss 기반 결과 (Base vs Exp-1) ===
base_loss_path = Path(LF_ROOT) / "outputs" / "stage1_eval" / "eval_loss" / "base" / "eval_results.json"
exp1_loss_path = Path(LF_ROOT) / "outputs" / "stage1_eval" / "eval_loss" / "exp1" / "eval_results.json"

if base_loss_path.exists() and exp1_loss_path.exists():
    with open(base_loss_path, 'r') as f:
        base_loss_metrics = json.load(f)
    with open(exp1_loss_path, 'r') as f:
        exp1_loss_metrics = json.load(f)
    print("=== Loss-based Evaluation (Base vs Exp-1) ===")
    print(f"  Base     eval_loss: {base_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(base_loss_metrics['eval_loss']):.4f}")
    print(f"  Exp-1    eval_loss: {exp1_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(exp1_loss_metrics['eval_loss']):.4f}")
    print()
else:
    print("[SKIP] Loss evaluation results not found\n")

# === 2. 생성 비교 결과 (모든 모델) ===
all_stage1_metrics = {}

for model_name, pred_path in MODEL_PREDS.items():
    if not pred_path.exists():
        print(f"[SKIP] {model_name}: Prediction results not found")
        continue

    metrics, results = evaluate_stage1_predictions(str(test_path), str(pred_path))
    all_stage1_metrics[model_name] = metrics

    print(f"\n=== {model_name} - Generation Evaluation ===")
    summary = pd.DataFrame({
        'Metric': [
            'BLEU-4', 'ROUGE-L', 'Exact Match',
            'Hungarian EA', 'Hungarian F1', 'Hungarian Prec',
            'Hungarian Rec', 'Hungarian Text', 'Hungarian Idx',
        ],
        'Score': [
            f"{metrics['avg_bleu']:.4f}",
            f"{metrics['avg_rouge_l']:.4f}",
            f"{metrics['exact_match_rate']:.1%}",
            f"{metrics['avg_hungarian_ea']:.4f}",
            f"{metrics['avg_hungarian_f1']:.4f}",
            f"{metrics['avg_hungarian_prec']:.4f}",
            f"{metrics['avg_hungarian_rec']:.4f}",
            f"{metrics['avg_hungarian_text']:.4f}",
            f"{metrics['avg_hungarian_idx']:.4f}",
        ]
    })
    display(summary)

# === 3. 비교 테이블 ===
if len(all_stage1_metrics) > 1:
    print("\n=== Stage 1 Comparison ===")
    comparison = pd.DataFrame({
        name: {
            'BLEU-4': f"{m['avg_bleu']:.4f}",
            'ROUGE-L': f"{m['avg_rouge_l']:.4f}",
            'Exact Match': f"{m['exact_match_rate']:.1%}",
            'Hungarian EA': f"{m['avg_hungarian_ea']:.4f}",
            'Hungarian F1': f"{m['avg_hungarian_f1']:.4f}",
            'Hungarian Prec': f"{m['avg_hungarian_prec']:.4f}",
            'Hungarian Rec': f"{m['avg_hungarian_rec']:.4f}",
            'Hungarian Text': f"{m['avg_hungarian_text']:.4f}",
            'Hungarian Idx': f"{m['avg_hungarian_idx']:.4f}",
        }
        for name, m in all_stage1_metrics.items()
    })
    display(comparison)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_names = list(all_stage1_metrics.keys())
n_models = len(model_names)
colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# === 1. Text Generation Metrics ===
text_metrics = ['avg_bleu', 'avg_rouge_l', 'exact_match_rate']
text_labels = ['BLEU-4', 'ROUGE-L', 'Exact Match']
x1 = np.arange(len(text_metrics))
width = 0.8 / n_models

for i, name in enumerate(model_names):
    values = [all_stage1_metrics[name][m] for m in text_metrics]
    offset = (i - n_models / 2 + 0.5) * width
    axes[0].bar(x1 + offset, values, width, label=name, color=colors[i])

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('5B. Text Generation Metrics')
axes[0].set_xticks(x1)
axes[0].set_xticklabels(text_labels, rotation=15)
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis='y', alpha=0.3)

# === 2. Hungarian Metrics ===
hung_metrics = ['avg_hungarian_ea', 'avg_hungarian_f1', 'avg_hungarian_prec',
                'avg_hungarian_rec', 'avg_hungarian_text', 'avg_hungarian_idx']
hung_labels = ['EA', 'F1', 'Precision', 'Recall', 'Text Sim', 'Index Acc']
x2 = np.arange(len(hung_metrics))

for i, name in enumerate(model_names):
    values = [all_stage1_metrics[name][m] for m in hung_metrics]
    offset = (i - n_models / 2 + 0.5) * width
    axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])

axes[1].set_xlabel('Metric')
axes[1].set_ylabel('Score')
axes[1].set_title('5B. Hungarian Element Matching')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(hung_labels, rotation=15)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
save_path = os.path.join(LF_ROOT, 'outputs', 'stage1_eval', 'hungarian_matching', 'stage1_hungarian_matching_evaluation.png')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n[Saved] {save_path}")

In [ ]:
import json
from pathlib import Path
from datetime import datetime

def generate_stage1_eval_report(model_name, metrics, output_dir, loss_metrics=None, comparison_metrics=None):
    """Stage 1 모델별 평가 결과를 Markdown 리포트로 생성하여 저장합니다."""
    m = metrics
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    lines = []
    lines.append(f"# Stage 1 - 5B. Hungarian Matching Report: {model_name}")
    lines.append(f"\n> Generated: {now}\n")

    # --- 실험 설정 ---
    lines.append("## Experiment Setup\n")
    lines.append("| Item | Value |")
    lines.append("|------|-------|")
    lines.append(f"| Model | {model_name} |")
    lines.append(f"| Test Samples | {m['total']} |")
    lines.append(f"| Task | World Modeling (UI State XML + Action → Next UI State XML) |")
    lines.append(f"| Method | Full Fine-tuning (Qwen3-VL-8B) |")
    lines.append(f"| Training | 3 epochs, LR=1e-5 (cosine), batch=32, RTX5090x2 |")
    lines.append("")

    # --- Loss-based Metrics (있을 경우) ---
    if loss_metrics:
        eval_loss = loss_metrics.get('eval_loss', 0)
        lines.append("## Loss-based Metrics\n")
        lines.append("| Metric | Score |")
        lines.append("|--------|-------|")
        lines.append(f"| Eval Loss | {eval_loss:.4f} |")
        lines.append(f"| Perplexity | {math.exp(eval_loss):.4f} |")
        lines.append("")

    # --- Overall Metrics ---
    lines.append("## Overall Metrics\n")
    lines.append("| Metric | Score |")
    lines.append("|--------|-------|")
    lines.append(f"| BLEU-4 | {m['avg_bleu']:.4f} |")
    lines.append(f"| ROUGE-L | {m['avg_rouge_l']:.4f} |")
    lines.append(f"| Exact Match | {m['exact_match_rate']:.1%} |")
    lines.append(f"| Hungarian EA | {m['avg_hungarian_ea']:.4f} |")
    lines.append(f"| Hungarian F1 | {m['avg_hungarian_f1']:.4f} |")
    lines.append(f"| Hungarian Precision | {m['avg_hungarian_prec']:.4f} |")
    lines.append(f"| Hungarian Recall | {m['avg_hungarian_rec']:.4f} |")
    lines.append(f"| Hungarian Text Sim | {m['avg_hungarian_text']:.4f} |")
    lines.append(f"| Hungarian Index Acc | {m['avg_hungarian_idx']:.4f} |")
    lines.append("")

    # --- Model Comparison (있을 경우) ---
    if comparison_metrics:
        lines.append("## Model Comparison\n")
        lines.append("| Metric | " + " | ".join(comparison_metrics.keys()) + " |")
        lines.append("|--------|" + "|".join(["-------"] * len(comparison_metrics)) + "|")

        metric_rows = [
            ("BLEU-4", "avg_bleu", ".4f"),
            ("ROUGE-L", "avg_rouge_l", ".4f"),
            ("Exact Match", "exact_match_rate", ".1%"),
            ("Hungarian EA", "avg_hungarian_ea", ".4f"),
            ("Hungarian F1", "avg_hungarian_f1", ".4f"),
            ("Hungarian Prec", "avg_hungarian_prec", ".4f"),
            ("Hungarian Rec", "avg_hungarian_rec", ".4f"),
            ("Hungarian Text", "avg_hungarian_text", ".4f"),
            ("Hungarian Idx", "avg_hungarian_idx", ".4f"),
        ]
        for label, key, fmt in metric_rows:
            values = []
            scores = [cm[key] for cm in comparison_metrics.values()]
            best = max(scores)
            for name, cm in comparison_metrics.items():
                val = cm[key]
                formatted = f"{val:{fmt}}"
                if val == best and len(set(scores)) > 1:
                    formatted = f"**{formatted}**"
                values.append(formatted)
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        lines.append("")

    # --- Metric Definitions ---
    lines.append("## Metric Definitions\n")
    lines.append("| Metric | Description |")
    lines.append("|--------|-------------|")
    lines.append("| BLEU-4 | 4-gram 기반 텍스트 생성 정확도 (0~1) |")
    lines.append("| ROUGE-L | LCS 기반 텍스트 유사도 F1 (0~1) |")
    lines.append("| Exact Match | 생성 텍스트가 정답과 완전 일치하는 비율 |")
    lines.append("| Hungarian EA | 헝가리안 매칭 기반 Element Accuracy (matched / max(pred, gt)) |")
    lines.append("| Hungarian F1 | 헝가리안 매칭 Precision과 Recall의 조화평균 |")
    lines.append("| Hungarian Precision | 예측 요소 중 정답과 매칭된 비율 |")
    lines.append("| Hungarian Recall | 정답 요소 중 예측과 매칭된 비율 |")
    lines.append("| Hungarian Text Sim | 매칭된 요소 쌍의 텍스트 Jaccard 유사도 평균 |")
    lines.append("| Hungarian Index Acc | 매칭된 요소 쌍의 DOM 위치 정확도 (차이 <= 2) |")
    lines.append("")

    report = "\n".join(lines)

    # --- 저장 ---
    output_path = Path(output_dir) / "evaluation_report.md"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)

    return str(output_path)

# === 각 모델별 리포트 생성 및 저장 ===
OUTPUT_DIRS = {
    "Baseline (Zero-shot)": Path(LF_ROOT) / "outputs" / "stage1_eval" / "hungarian_matching" / "base",
    "Exp-1 (Fine-tuned)": Path(LF_ROOT) / "outputs" / "stage1_eval" / "hungarian_matching" / "full_world_model",
}

# Loss 결과 로드 (모델별 매핑)
loss_metrics_map = {}
base_loss_path = Path(LF_ROOT) / "outputs" / "stage1_eval" / "eval_loss" / "base" / "eval_results.json"
exp1_loss_path = Path(LF_ROOT) / "outputs" / "stage1_eval" / "eval_loss" / "exp1" / "eval_results.json"

if base_loss_path.exists():
    with open(base_loss_path, 'r') as f:
        loss_metrics_map["Baseline (Zero-shot)"] = json.load(f)
if exp1_loss_path.exists():
    with open(exp1_loss_path, 'r') as f:
        loss_metrics_map["Exp-1 (Fine-tuned)"] = json.load(f)

for model_name, m in all_stage1_metrics.items():
    output_dir = OUTPUT_DIRS.get(model_name)
    if output_dir:
        lm = loss_metrics_map.get(model_name)
        saved = generate_stage1_eval_report(
            model_name=model_name,
            metrics=m,
            output_dir=output_dir,
            loss_metrics=lm,
            comparison_metrics=all_stage1_metrics,
        )
        print(f"[Saved] {saved}")

print("\nDone.")

## 6. Stage 2 SFT Training

**핵심 가설**: GUI World Modeling으로 사전학습된 모델은 동일 베이스 대비 Action Prediction SFT에서 더 높은 성능을 보인다.

| ID | 역할 | 모델 | HuggingFace ID |
|----|------|------|----------------|
| [Exp-2] | Base | Qwen3-VL-8B-Instruct | `Qwen/Qwen3-VL-8B-Instruct` |
| [Exp-3] | World Model | Stage 1 Merged | `SaFD-00/qwen3-vl-8b-stage1-world-model` |
| [Exp-4] | gWorld | gWorld-8B | `trillionlabs/gWorld-8B` |

**Base 비교:** Qwen3-VL-8B-Instruct (Zero-shot, 학습 없음)도 함께 평가

**핵심 비교:**
- Exp-2 vs Exp-3 → World Modeling 사전학습이 Action Prediction에 미치는 효과
- Exp-3 vs Exp-4 → 자체 World Model vs 기존 World Model(gWorld)
- Exp-2 vs Exp-4 → Base vs gWorld

**공통 학습 설정:**

| 항목 | 값 |
|------|-----|
| Method | LoRA (r=16, α=32, dropout=0.1) |
| Dataset | GUI-Model_stage2_train (~2,990건, 95% split) |
| Effective Batch | 32 (2 × 8 × 2 GPU) |
| Learning Rate | 5e-5 (cosine, warmup=0.05) |
| Epochs | 1.0 |
| Precision | bf16 |
| Hardware | RTX 5090 32GB × 2 |

In [ ]:
import os
from pathlib import Path

# === Stage 2 YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage2_lora"
yaml_dir.mkdir(parents=True, exist_ok=True)

# 공통 설정 템플릿
COMMON_CONFIG = """\
### model
model_name_or_path: {model_name_or_path}
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: true
lora_rank: 16
lora_alpha: 32
lora_target: all
lora_dropout: 0.1

### dataset
dataset: GUI-Model_stage2_train
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/{output_dir}
logging_steps: 1
save_strategy: "no"
# save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 5.0e-5
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
"""

MODELS = {
    "base.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "output_dir": "stage2_lora/lora_base",
    },
    "world_model.yaml": {
        "model_name_or_path": "SaFD-00/qwen3-vl-8b-stage1-world-model",
        "output_dir": "stage2_lora/lora_world_model",
    },
    "gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "output_dir": "stage2_lora/lora_gworld",
    },
}

for fname, params in MODELS.items():
    content = COMMON_CONFIG.format(
        model_name_or_path=params["model_name_or_path"],
        output_dir=params["output_dir"],
    )
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  base: {params['model_name_or_path']}")
    print(f"  output: ./outputs/{params['output_dir']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [Exp-2] Stage 2 - Base (Qwen3-VL-8B-Instruct) | RTX 5090 x 2
!llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Exp-3] Stage 2 - World Model (Stage 1 Merged) | RTX 5090 x 2
!llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/world_model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Exp-4] Stage 2 - gWorld-8B (World Model) | RTX 5090 x 2
!llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/gworld.yaml

## 7. Stage 2 Merge & Upload to HuggingFace

Stage 2 LoRA 어댑터를 베이스 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**Merge 대상:**

| ID | Base Model | Adapter | Export |
|----|------------|---------|--------|
| [Exp-2] | `Qwen/Qwen3-VL-8B-Instruct` | `./outputs/stage2_lora/lora_base` | `SaFD-00/qwen3-vl-8b-stage2-base` |
| [Exp-3] | `SaFD-00/qwen3-vl-8b-stage1-world-model` | `./outputs/stage2_lora/lora_world_model` | `SaFD-00/qwen3-vl-8b-stage2-world-model` |
| [Exp-4] | `trillionlabs/gWorld-8B` | `./outputs/stage2_lora/lora_gworld` | `SaFD-00/qwen3-vl-8b-stage2-gworld` |

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
from pathlib import Path

# === Stage 2 Merge YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "stage2"
yaml_dir.mkdir(parents=True, exist_ok=True)

MERGE_TEMPLATE = """\
### model
model_name_or_path: {model_name_or_path}
adapter_name_or_path: {adapter_path}
trust_remote_code: true
finetuning_type: lora
template: qwen3_vl_nothink

### export
export_dir: ./exports/{export_dir}
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {hub_model_id}
"""

MERGE_MODELS = {
    "merge_base.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "adapter_path": "./outputs/stage2_lora/lora_base",
        "export_dir": "qwen3-vl-8b-stage2-base",
        "hub_model_id": "SaFD-00/qwen3-vl-8b-stage2-base",
    },
    "merge_world_model.yaml": {
        "model_name_or_path": "SaFD-00/qwen3-vl-8b-stage1-world-model",
        "adapter_path": "./outputs/stage2_lora/lora_world_model",
        "export_dir": "qwen3-vl-8b-stage2-world-model",
        "hub_model_id": "SaFD-00/qwen3-vl-8b-stage2-world-model",
    },
    "merge_gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "adapter_path": "./outputs/stage2_lora/lora_gworld",
        "export_dir": "qwen3-vl-8b-stage2-gworld",
        "hub_model_id": "SaFD-00/qwen3-vl-8b-stage2-gworld",
    },
}

for fname, params in MERGE_MODELS.items():
    content = MERGE_TEMPLATE.format(**params)
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  base: {params['model_name_or_path']}")
    print(f"  adapter: {params['adapter_path']}")
    print(f"  export: {params['hub_model_id']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [Exp-2] Merge - Base (Qwen3-VL-8B-Instruct)
!llamafactory-cli export examples/merge_custom/GUI-Model/stage2/merge_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Exp-3] Merge - World Model (Stage 1 Merged + Stage 2 LoRA)
!llamafactory-cli export examples/merge_custom/GUI-Model/stage2/merge_world_model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Exp-4] Merge - gWorld-8B
!llamafactory-cli export examples/merge_custom/GUI-Model/stage2/merge_gworld.yaml

## 8. Stage 2 Evaluation Pipeline

Test set(`GUI-Model_stage2_test`)을 사용하여 4개 모델(Exp-2, Exp-3, Exp-4 + Base Zero-shot)의 Action Prediction 성능을 비교합니다.

**파이프라인:**
1. Prediction YAML 생성 (3모델 + base zero-shot)
2. `vllm_infer.py`로 test set prediction 수행
3. 평가 함수로 action 단위 메트릭 계산
4. 모델별 비교 테이블 & 시각화

**평가 메트릭:**

| Metric | Formula | Description |
|--------|---------|-------------|
| Parse Rate | 유효 JSON / 전체 | 모델 출력 파싱 성공률 |
| Type Accuracy | 정확 type / 전체 | Action type 일치율 |
| Bounds IoU | IoU(GT, Pred) | Bounding box 겹침 비율 |
| Params Match | 정확 params / 전체 | Action parameters 일치율 |
| Overall Score | Type × (0.5×IoU + 0.5×Params) | 종합 점수 |

**데이터:** GUI-Model_stage2_test (~157건, 5% split)

In [ ]:
os.chdir(LF_ROOT)
!mkdir -p ./outputs/stage2_eval/base

## [Base Zero-shot] Prediction - Qwen3-VL-8B-Instruct (Zero-shot, no training)
!python scripts/vllm_infer.py \
    --model_name_or_path Qwen/Qwen3-VL-8B-Instruct \
    --dataset GUI-Model_stage2_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage2_eval/base/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage2_eval/base/predict_results.json

In [ ]:
os.chdir(LF_ROOT)
!mkdir -p ./outputs/stage2_eval/lora_base

## [Exp-2] Prediction - Base (Qwen3-VL-8B-Instruct + LoRA) | RTX 5090 x 2
!python scripts/vllm_infer.py \
    --model_name_or_path SaFD-00/qwen3-vl-8b-stage2-base \
    --dataset GUI-Model_stage2_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage2_eval/lora_base/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage2_eval/lora_base/predict_results.json

In [ ]:
os.chdir(LF_ROOT)
!mkdir -p ./outputs/stage2_eval/lora_world_model

## [Exp-3] Prediction - World Model (Stage 1 Merged + LoRA) | RTX 5090 x 2
!python scripts/vllm_infer.py \
    --model_name_or_path SaFD-00/qwen3-vl-8b-stage2-world-model \
    --dataset GUI-Model_stage2_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage2_eval/lora_world_model/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage2_eval/lora_world_model/predict_results.json

In [ ]:
os.chdir(LF_ROOT)
!mkdir -p ./outputs/stage2_eval/lora_gworld

## [Exp-4] Prediction - gWorld-8B (World Model + LoRA) | RTX 5090 x 2
!python scripts/vllm_infer.py \
    --model_name_or_path SaFD-00/qwen3-vl-8b-stage2-gworld \
    --dataset GUI-Model_stage2_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage2_eval/lora_gworld/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage2_eval/lora_gworld/predict_results.json

In [ ]:
import json
import re
from collections import defaultdict

def parse_bounds(bounds_str):
    """Parse bounds string '[x1, y1, x2, y2]' to list of floats."""
    try:
        nums = re.findall(r'[\d.]+', str(bounds_str))
        return [float(n) for n in nums[:4]] if len(nums) >= 4 else None
    except:
        return None

def calc_iou(box1, box2):
    """Calculate IoU between two bounding boxes [x1, y1, x2, y2]."""
    if box1 is None or box2 is None:
        return 0.0
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def parse_action(text):
    """Parse action JSON from model output text."""
    text = text.strip()
    # Try direct JSON parse
    try:
        return json.loads(text)
    except:
        pass
    # Try extracting JSON from markdown code block
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            pass
    # Try finding first JSON object
    match = re.search(r'\{[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    return None

def evaluate_single(gt_action, pred_action):
    """Evaluate a single prediction against ground truth."""
    result = {'type_correct': False, 'iou': 0.0, 'params_correct': False, 'has_bounds': False, 'has_params': False}

    if pred_action is None:
        return result

    gt_type = gt_action.get('type', '')
    pred_type = pred_action.get('type', '')
    result['type_correct'] = (gt_type.lower() == pred_type.lower())

    # Bounds IoU
    gt_bounds = parse_bounds(gt_action.get('bounds'))
    pred_bounds = parse_bounds(pred_action.get('bounds'))
    if gt_bounds is not None:
        result['has_bounds'] = True
        result['iou'] = calc_iou(gt_bounds, pred_bounds)

    # Params match
    param_keys = {'openapp': 'app', 'input': 'text', 'swipe': 'direction'}
    if gt_type.lower() in param_keys:
        result['has_params'] = True
        key = param_keys[gt_type.lower()]
        gt_val = str(gt_action.get(key, '')).strip().lower()
        pred_val = str(pred_action.get(key, '')).strip().lower()
        result['params_correct'] = (gt_val == pred_val)
    elif gt_type.lower() not in ['click', 'long click']:
        # Types without bounds or special params
        result['has_params'] = False

    return result

def evaluate_predictions(test_path, pred_path):
    """Evaluate all predictions and return metrics."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]

    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    per_type = defaultdict(lambda: {'count': 0, 'type_correct': 0, 'iou_sum': 0.0, 'iou_count': 0, 'params_correct': 0, 'params_count': 0})

    for i, (gt_entry, pred_entry) in enumerate(zip(gt_entries, pred_entries)):
        gt_action = json.loads(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))
        pred_action = parse_action(pred_text)

        r = evaluate_single(gt_action, pred_action)
        r['gt_type'] = gt_action.get('type', 'unknown').lower()
        results.append(r)

        t = r['gt_type']
        per_type[t]['count'] += 1
        per_type[t]['type_correct'] += int(r['type_correct'])
        if r['has_bounds']:
            per_type[t]['iou_sum'] += r['iou']
            per_type[t]['iou_count'] += 1
        if r['has_params']:
            per_type[t]['params_correct'] += int(r['params_correct'])
            per_type[t]['params_count'] += 1

    total = len(results)
    parsed = sum(1 for r in results if r['type_correct'] or r.get('gt_type'))
    type_correct = sum(r['type_correct'] for r in results)
    bounds_entries = [r for r in results if r['has_bounds']]
    params_entries = [r for r in results if r['has_params']]

    avg_iou = sum(r['iou'] for r in results) / total if total else 0
    cond_iou = sum(r['iou'] for r in bounds_entries) / len(bounds_entries) if bounds_entries else 0
    params_acc = sum(r['params_correct'] for r in params_entries) / len(params_entries) if params_entries else 0
    cond_params = params_acc
    type_acc = type_correct / total if total else 0
    overall = type_acc * (0.5 * avg_iou + 0.5 * (sum(r['params_correct'] for r in results) / total if total else 0))

    per_type_summary = {}
    for t, d in per_type.items():
        per_type_summary[t] = {
            'count': d['count'],
            'type_acc': d['type_correct'] / d['count'] if d['count'] else 0,
            'avg_iou': d['iou_sum'] / d['iou_count'] if d['iou_count'] else 0,
            'params_acc': d['params_correct'] / d['params_count'] if d['params_count'] else 0,
        }

    metrics = {
        'total': total,
        'parse_rate': parsed / total if total else 0,
        'type_accuracy': type_acc,
        'avg_bounds_iou': avg_iou,
        'cond_bounds_iou': cond_iou,
        'params_accuracy': sum(r['params_correct'] for r in results) / total if total else 0,
        'cond_params_acc': cond_params,
        'overall_score': overall,
        'per_type': per_type_summary,
    }

    return metrics, results

print("[Loaded] Evaluation functions: parse_bounds, calc_iou, parse_action, evaluate_single, evaluate_predictions")

In [ ]:
import json
import pandas as pd
from pathlib import Path

test_path = Path(LF_ROOT) / "data" / "GUI-Model" / "gui-model_stage2_test.jsonl"

MODEL_PREDS = {
    "Base (Zero-shot)": Path(LF_ROOT) / "outputs" / "stage2_eval" / "base" / "generated_predictions.jsonl",
    "Exp-2: Base": Path(LF_ROOT) / "outputs" / "stage2_eval" / "lora_base" / "generated_predictions.jsonl",
    "Exp-3: World Model": Path(LF_ROOT) / "outputs" / "stage2_eval" / "lora_world_model" / "generated_predictions.jsonl",
    "Exp-4: gWorld-8B": Path(LF_ROOT) / "outputs" / "stage2_eval" / "lora_gworld" / "generated_predictions.jsonl",
}

all_metrics = {}
all_results = {}

for model_name, pred_path in MODEL_PREDS.items():
    if not pred_path.exists():
        print(f"[SKIP] {model_name}: {pred_path.name} not found")
        continue
    metrics, results = evaluate_predictions(str(test_path), str(pred_path))
    all_metrics[model_name] = metrics
    all_results[model_name] = results
    print(f"[Done] {model_name}")

# === Summary Table ===
if all_metrics:
    summary_df = pd.DataFrame({
        name: {
            'Parse Rate': f"{m['parse_rate']:.1%}",
            'Type Accuracy': f"{m['type_accuracy']:.1%}",
            'Bounds IoU': f"{m['avg_bounds_iou']:.3f}",
            'Params Accuracy': f"{m['params_accuracy']:.1%}",
            'Cond. IoU': f"{m['cond_bounds_iou']:.3f}",
            'Cond. Params': f"{m['cond_params_acc']:.1%}",
            'Overall Score': f"{m['overall_score']:.3f}",
        }
        for name, m in all_metrics.items()
    }).T

    print("\n" + "="*70)
    print("=== Stage 2 Model Comparison (4-Way) ===")
    print("="*70)
    display(summary_df)

    # === Per-Type Table ===
    for model_name, m in all_metrics.items():
        print(f"\n--- {model_name} Per-Type ---")
        type_df = pd.DataFrame(m['per_type']).T
        type_df.columns = ['Count', 'Type Acc', 'Avg IoU', 'Params Acc']
        display(type_df)
else:
    print("[WARN] No prediction results found. Run prediction cells first.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_names = list(all_metrics.keys())
metrics_to_plot = ['type_accuracy', 'avg_bounds_iou', 'params_accuracy', 'overall_score']
metric_labels = ['Type Accuracy', 'Bounds IoU', 'Params Accuracy', 'Overall Score']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# === 1. Bar Chart: 모델별 메트릭 비교 ===
x = np.arange(len(metrics_to_plot))
n_models = len(model_names)
width = 0.8 / n_models
colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]

for i, name in enumerate(model_names):
    values = [all_metrics[name][m] for m in metrics_to_plot]
    offset = (i - n_models / 2 + 0.5) * width
    axes[0].bar(x + offset, values, width, label=name, color=colors[i])

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Stage 2: 4-Way Model Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_labels, rotation=15)
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis='y', alpha=0.3)

# === 2. Per-Type Accuracy Comparison ===
action_types = sorted(set(
    t for m in all_metrics.values() for t in m['per_type'].keys()
))
x2 = np.arange(len(action_types))

for i, name in enumerate(model_names):
    values = [all_metrics[name]['per_type'].get(t, {}).get('type_acc', 0) for t in action_types]
    offset = (i - n_models / 2 + 0.5) * width
    axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])

axes[1].set_xlabel('Action Type')
axes[1].set_ylabel('Type Accuracy')
axes[1].set_title('Per-Type Accuracy Comparison')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(action_types, rotation=30, fontsize=8)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LF_ROOT, 'outputs', 'stage2_eval', 'stage2_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n[Saved] {os.path.join(LF_ROOT, 'outputs', 'stage2_eval', 'stage2_evaluation.png')}")

In [ ]:
import json
from pathlib import Path
from datetime import datetime

def generate_eval_report(model_name, metrics, results, output_dir, comparison_metrics=None):
    """모델별 평가 결과를 Markdown 리포트로 생성하여 저장합니다."""
    m = metrics
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    lines = []
    lines.append(f"# Stage 2 Evaluation Report: {model_name}")
    lines.append(f"\n> Generated: {now}\n")

    # --- 실험 설정 ---
    lines.append("## Experiment Setup\n")
    lines.append("| Item | Value |")
    lines.append("|------|-------|")
    lines.append(f"| Model | {model_name} |")
    lines.append(f"| Test Samples | {m['total']} |")
    lines.append(f"| Task | Action Prediction (Screenshot + XML + Task → Action JSON) |")
    lines.append(f"| Method | LoRA (r=16, α=32, dropout=0.1) |")
    lines.append(f"| Training | 1 epoch, LR=5e-5 (cosine), batch=32, RTX5090x2 |")
    lines.append("")

    # --- Overall Metrics ---
    lines.append("## Overall Metrics\n")
    lines.append("| Metric | Score |")
    lines.append("|--------|-------|")
    lines.append(f"| Parse Rate | {m['parse_rate']:.1%} |")
    lines.append(f"| Type Accuracy | {m['type_accuracy']:.1%} |")
    lines.append(f"| Bounds IoU (avg) | {m['avg_bounds_iou']:.4f} |")
    lines.append(f"| Bounds IoU (conditional) | {m['cond_bounds_iou']:.4f} |")
    lines.append(f"| Params Accuracy (avg) | {m['params_accuracy']:.1%} |")
    lines.append(f"| Params Accuracy (conditional) | {m['cond_params_acc']:.1%} |")
    lines.append(f"| **Overall Score** | **{m['overall_score']:.4f}** |")
    lines.append("")

    # --- Per-Type Breakdown ---
    lines.append("## Per-Type Breakdown\n")
    lines.append("| Action Type | Count | Type Acc | Avg IoU | Params Acc |")
    lines.append("|-------------|-------|----------|---------|------------|")
    for t, d in sorted(m['per_type'].items(), key=lambda x: -x[1]['count']):
        lines.append(f"| {t} | {d['count']} | {d['type_acc']:.1%} | {d['avg_iou']:.4f} | {d['params_acc']:.1%} |")
    lines.append("")

    # --- Comparison (있을 경우) ---
    if comparison_metrics:
        lines.append("## Model Comparison\n")
        lines.append("| Metric | " + " | ".join(comparison_metrics.keys()) + " |")
        lines.append("|--------|" + "|".join(["-------"] * len(comparison_metrics)) + "|")

        metric_rows = [
            ("Parse Rate", "parse_rate", ".1%"),
            ("Type Accuracy", "type_accuracy", ".1%"),
            ("Bounds IoU", "avg_bounds_iou", ".4f"),
            ("Cond. IoU", "cond_bounds_iou", ".4f"),
            ("Params Accuracy", "params_accuracy", ".1%"),
            ("Cond. Params", "cond_params_acc", ".1%"),
            ("Overall Score", "overall_score", ".4f"),
        ]
        for label, key, fmt in metric_rows:
            values = []
            scores = [cm[key] for cm in comparison_metrics.values()]
            best = max(scores)
            for name, cm in comparison_metrics.items():
                val = cm[key]
                formatted = f"{val:{fmt}}"
                if val == best and len(set(scores)) > 1:
                    formatted = f"**{formatted}**"
                values.append(formatted)
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        lines.append("")

    # --- Metric Definitions ---
    lines.append("## Metric Definitions\n")
    lines.append("| Metric | Description |")
    lines.append("|--------|-------------|")
    lines.append("| Parse Rate | 모델 출력이 유효한 JSON으로 파싱된 비율 |")
    lines.append("| Type Accuracy | 예측된 action type이 GT와 일치하는 비율 |")
    lines.append("| Bounds IoU (avg) | 전체 샘플 대상 bounding box IoU 평균 (bounds 없는 타입은 0) |")
    lines.append("| Bounds IoU (conditional) | bounds가 있는 샘플만 대상 IoU 평균 |")
    lines.append("| Params Accuracy (avg) | 전체 샘플 대상 파라미터 정확도 (params 없는 타입은 0) |")
    lines.append("| Params Accuracy (conditional) | params가 있는 샘플만 대상 정확도 |")
    lines.append("| Overall Score | Type_Acc × (0.5 × Avg_IoU + 0.5 × Avg_Params) |")
    lines.append("")

    report = "\n".join(lines)

    # --- 저장 ---
    output_path = Path(output_dir) / "evaluation_report.md"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)

    return str(output_path)

# === 각 모델별 리포트 생성 및 저장 ===
OUTPUT_DIRS = {
    "Base (Zero-shot)": Path(LF_ROOT) / "outputs" / "stage2_eval" / "base",
    "Exp-2: Base": Path(LF_ROOT) / "outputs" / "stage2_eval" / "lora_base",
    "Exp-3: World Model": Path(LF_ROOT) / "outputs" / "stage2_eval" / "lora_world_model",
    "Exp-4: gWorld-8B": Path(LF_ROOT) / "outputs" / "stage2_eval" / "lora_gworld",
}

for model_name, m in all_metrics.items():
    output_dir = OUTPUT_DIRS.get(model_name)
    if output_dir:
        saved = generate_eval_report(
            model_name=model_name,
            metrics=m,
            results=all_results[model_name],
            output_dir=output_dir,
            comparison_metrics=all_metrics,
        )
        print(f"[Saved] {saved}")

print("\nDone.")